## Execução de Código Baseada em Agente usando Amazon AgentCore Bedrock Code Interpreter - Tutorial(Langchain)
Este tutorial demonstra como criar um agente de IA que valida respostas através da execução de código usando Python. Usamos o Amazon Bedrock AgentCore Code Interpreter para executar código que é gerado pelo LLM

Este tutorial demonstra como usar o AgentCore Bedrock Code Interpreter para:
1. Configurar um ambiente sandbox
2. Configurar um agente baseado em langchain que gera código com base na consulta do usuário
3. Executar código em um ambiente sandbox usando Code Interpreter
4. Exibir os resultados de volta ao usuário

## Pré-requisitos
- Conta AWS com acesso ao Bedrock AgentCore Code Interpreter
- Você possui as permissões IAM necessárias para criar e gerenciar recursos do code interpreter
- Pacotes Python necessários instalados (incluindo boto3, bedrock-agentcore e langchain)
- A função IAM deve ter permissões para invocar modelos no Amazon Bedrock
 - Acesso ao modelo Claude 3.5 Sonnet na região US Oregon (us-west-2)

## Sua função de execução IAM deve ter a seguinte política IAM anexada



~~~ {
"Version": "2012-10-17",
"Statement": [
    {
        "Effect": "Allow",
        "Action": [
            "bedrock-agentcore:CreateCodeInterpreter",
            "bedrock-agentcore:StartCodeInterpreterSession",
            "bedrock-agentcore:InvokeCodeInterpreter",
            "bedrock-agentcore:StopCodeInterpreterSession",
            "bedrock-agentcore:DeleteCodeInterpreter",
            "bedrock-agentcore:ListCodeInterpreters",
            "bedrock-agentcore:GetCodeInterpreter"
        ],
        "Resource": "*"
    },
    {
        "Effect": "Allow",
        "Action": [
            "logs:CreateLogGroup",
            "logs:CreateLogStream",
            "logs:PutLogEvents"
        ],
        "Resource": "arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/code-interpreter*"
    }
]
}

## Como funciona

O sandbox de execução de código permite que agentes processem consultas de usuários de forma segura, criando um ambiente isolado com um interpretador de código, shell e sistema de arquivos. Após um Large Language Model ajudar com a seleção de ferramentas, o código é executado dentro desta sessão, antes de ser retornado ao usuário ou agente para síntese.

![architecture local](code-interpreter.png)

## 1. Configurando o Ambiente

Primeiro, vamos importar as bibliotecas necessárias e inicializar nossa sessão do Code Interpreter.

In [ ]:
!pip install --upgrade -r requirements.txt

In [17]:
from bedrock_agentcore.tools.code_interpreter_client import code_session
from langchain.agents import AgentExecutor, create_tool_calling_agent, initialize_agent, tool
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import json


## 2. Definição do Prompt do Sistema
Defina o comportamento e as capacidades do assistente de IA. Instruímos nosso assistente a sempre validar respostas através da execução de código e raciocínio baseado em dados.

In [8]:
SYSTEM_PROMPT = """Você é um assistente de IA útil que valida todas as respostas através da execução de código.

PRINCÍPIOS DE VALIDAÇÃO:
1. Ao fazer afirmações sobre código, algoritmos ou cálculos - escreva código para verificá-los
2. Use execute_python para testar cálculos matemáticos, algoritmos e lógica
3. Crie scripts de teste para validar sua compreensão antes de dar respostas
4. Sempre mostre seu trabalho com a execução real do código
5. Se incerto, declare explicitamente as limitações e valide o que você pode

ABORDAGEM:
- Se perguntado sobre um conceito de programação, implemente-o em código para demonstrar
- Se perguntado sobre cálculos, compute-os programaticamente E mostre o código
- Se implementando algoritmos, inclua casos de teste para provar a correção
- Documente seu processo de validação para transparência
- O sandbox mantém o estado entre as execuções, então você pode se referir a resultados anteriores

FERRAMENTA DISPONÍVEL:
- execute_python: Executar código Python e ver a saída

FORMATO DE RESPOSTA: A ferramenta execute_python retorna uma resposta JSON com:
- sessionId: O ID da sessão sandbox
- id: ID da requisição
- isError: Booleano indicando se houve um erro
- content: Array de objetos de conteúdo com tipo e texto/dados
- structuredContent: Para execução de código, inclui stdout, stderr, exitCode, executionTime"""

## 3. Definição da Ferramenta de Execução de Código
Em seguida, definimos a função como ferramenta que será usada pelo Agente como tool, para executar código no sandbox de código. Usamos o decorador @tool para anotar a função como uma ferramenta personalizada para o agente.

Dentro de uma sessão ativa do code interpreter, você pode executar código em linguagens suportadas (Python, JavaScript), acessar bibliotecas com base em sua configuração de dependências, gerar visualizações e manter o estado entre as execuções.

In [11]:
@tool
def execute_python(code: str, description: str = "") -> str:
    """Executar código Python no sandbox."""
    
    if description:
        code = f"# {description}\n{code}"
    
    print(f"\n Código Gerado: {code}")
    
    with code_session("us-west-2") as code_client:
        response = code_client.invoke("executeCode", {
            "code": code,
            "language": "python",
            "clearContext": False
        })
    
    for event in response["stream"]:
        return json.dumps(event["result"])

## 4. Configuração do Agente
Criamos e configuramos um agente usando o SDK langchain. Fornecemos a ele o prompt do sistema e a ferramenta que definimos acima para executar e gerar código

#### 4.1 Inicializar o modelo de linguagem

In [6]:
llm = ChatBedrockConverse(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",region_name="us-west-2")

#### 4.2 Definir o template de prompt

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

#### 4.3 Criar uma lista de nossas ferramentas personalizadas

In [ ]:
tools = [execute_python]

### 4.4 Criar o executor do agente


In [ ]:
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 5. Definição da Consulta
Defina uma consulta de exemplo para testar o agente com capacidades de execução de código

In [ ]:
query="Todos os planetas do sistema solar cabem entre a Terra e a Lua?"

## 6. Invocação do Agente e Processamento da Resposta
Invocamos o agente com nossa consulta e processamos a resposta do agente. Note que o agente realiza raciocínio começando com uma hipótese, valida convertendo-a em código e executa o código no code interpreter

In [ ]:
response=agent_executor.invoke({"input": query})

### A resposta final do nosso agente...

In [ ]:
print(response['output'][0]['text'])